# Notebook 29 — Training Embedding Models and Rerankers

    ## Learning objectives

    - Derive contrastive bi-encoder and cross-encoder objectives
- Mine hard negatives without poisoning labels
- Fine-tune and evaluate retrieval by slice

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ['sentence-transformers>=4,<6', 'datasets>=3.5,<6']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if True and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 29.1 Representation objectives

Bi-encoders embed query and document independently for scalable retrieval. Contrastive training raises positive similarity relative to negatives; multiple-negatives ranking uses other positives in the batch as negatives and is efficient only when false negatives are controlled. Triplet and margin losses encode relative constraints. Normalize vectors when cosine similarity is intended. Dataset pairs define relevance and can teach shortcuts from source, length, or formatting.


In [ ]:
import torch
q=torch.nn.functional.normalize(torch.randn(4,8),dim=-1); d=torch.nn.functional.normalize(torch.randn(4,8),dim=-1); logits=q@d.T/.07; labels=torch.arange(4); print(torch.nn.functional.cross_entropy(logits,labels))


## 29.2 Negatives and mining

Random negatives are often too easy; lexical or dense mining finds confusable candidates that teach decision boundaries. A highly ranked unlabeled document may actually be relevant, creating false-negative gradients. Use human judgments, cross-encoder filtering, known-positive sets, and iterative audits. Mine with a frozen prior model and keep evaluation documents isolated. Balance domains and query frequencies by effective pairs or tokens.


In [ ]:
pool=[{"id":"d1","score":.91,"known_positive":True},{"id":"d2","score":.89,"known_positive":False},{"id":"d3","score":.2,"known_positive":False}]
print("audit hard negatives",[x for x in pool if not x["known_positive"] and x["score"]>.5])


## 29.3 Cross-encoder rerankers

A cross-encoder jointly reads query and candidate and predicts relevance, capturing token interactions at greater latency. Train with pointwise classification, pairwise preference, or listwise ranking objectives. Calibrate scores only for the deployment candidate distribution. Reranker training cannot recover documents absent from first-stage candidates, so freeze candidate pools when evaluating reranker improvements and measure end-to-end recall separately.


In [ ]:
pointwise=torch.tensor([2.,-1.]); labels=torch.tensor([1.,0.]); print(torch.nn.functional.binary_cross_entropy_with_logits(pointwise,labels))


## 29.4 Training and evaluation

Sentence Transformers provides losses, trainers, evaluators, and Hub-compatible artifacts. Begin with a small domain dataset and guarded run; record base revision, pooling, maximum length, prompts, batch composition, negatives, and precision. Evaluate Recall@k, MRR, nDCG, MAP, latency, index size, multilingual/domain slices, and robustness to identifiers. Compare base and tuned models on a frozen corpus, rebuild the index after any embedding change, and retain rollback lineage.


In [ ]:
ranking=["d3","d1","d7"]; relevant={"d1","d2"}; print("recall@3",len(set(ranking)&relevant)/len(relevant),"RR",1/2)


## Reference workflow and evidence standard

Treat the notebook as an experiment, not a recipe. State the question, freeze inputs and
success criteria, establish the simplest baseline, change one material factor, and retain raw
outputs needed to diagnose failures. Record model, tokenizer, template, data and code revisions;
hardware and dtype; random seeds; generation or optimization configuration; token counts;
latency and memory; and results by meaningful slice. A demonstration that runs is evidence of
plumbing, not evidence of general capability.

Test boundaries as well as the happy path: empty and maximum-length inputs, malformed records,
multilingual or code text, unavailable dependencies, cancellation, and adversarial content.
Keep credentials in environment or Colab Secrets and never serialize them with artifacts. Pin
remote revisions, review licenses and custom code, validate saved artifacts in a fresh process,
and prefer deterministic validators wherever outputs can be checked mechanically.

Before applying the technique, compare it with prompting, retrieval, a smaller model, or no
model. Report quality together with compute, storage, latency, and operational complexity. Use
held-out data and paired comparisons, disclose uncertainty and negative results, and define a
rollback path. These practices connect low-level understanding to reliable application work.

A useful completion checklist asks four separate questions. Is the mathematical contract clear
enough to predict shapes, masks, reductions, and failure cases? Does the implementation reproduce
a tiny hand-worked or deterministic reference? Does the measured result survive a held-out set,
relevant slices, and an ablation against a simpler baseline? Can another person reload the exact
artifacts and reconstruct the claim from the manifest? Passing only the first two establishes a
tutorial demonstration; passing all four supports an engineering decision. When a result fails,
preserve the counterexample and update the test suite before changing the implementation.

Finally, separate correctness, capability, efficiency, and safety conclusions. A correct
implementation may have weak capability; a capable prototype may be too costly or unsafe to
deploy. Name the population to which each conclusion applies and avoid converting a single
metric into a universal ranking. Track assumptions beside results, especially tokenizer and
template compatibility, data rights, access-control boundaries, and hardware-specific behavior.
Leave exercises with an executable acceptance criterion, a baseline result, and a short written
interpretation. That combination turns exploratory code into cumulative course evidence that can
be revisited when libraries, model families, or deployment engines change.


## 29.5 Multiple-negatives contrastive loss

In-batch negatives make other documents in the batch serve as negatives for each query, producing an efficient similarity matrix. The method assumes those off-diagonal pairs are truly irrelevant; duplicates, related questions, and multi-positive cases create false-negative gradients. Normalize embeddings when using cosine-like scores, choose temperature deliberately, and distribute negatives consistently across workers. Track retrieval metrics, embedding norms, and false-negative audits rather than training loss alone.


In [ ]:
torch.manual_seed(1); q=torch.nn.functional.normalize(torch.randn(4,6),dim=-1); d=torch.nn.functional.normalize(q+.1*torch.randn(4,6),dim=-1); temperature=.07
sim=q@d.T/temperature; labels=torch.arange(4); contrastive=torch.nn.functional.cross_entropy(sim,labels); print(sim.argmax(-1),contrastive.item())


## 29.6 Reranker training and calibration

A cross-encoder reranker jointly reads query and document and predicts relevance, trading latency for interaction quality. Train on judged pairs or lists with pointwise, pairwise, or listwise objectives, preserving hard and random negatives. Evaluate reranking only after freezing the candidate set so gains are not confused with retrieval changes. Score scales are model-specific and not probabilities without calibration. Measure NDCG, MRR, Recall after reranking, latency per candidate, and failure slices; choose candidate depth from an end-to-end quality/latency curve.


In [ ]:
scores=torch.tensor([2.1,.4,1.7,-.2]); relevance=torch.tensor([3.,0.,2.,1.]); order=scores.argsort(descending=True); gains=(2**relevance[order]-1); discounts=1/torch.log2(torch.arange(2,6,dtype=torch.float)); dcg=(gains*discounts).sum(); ideal=((2**relevance.sort(descending=True).values-1)*discounts).sum(); print("NDCG",(dcg/ideal).item(),"order",order.tolist())


## Exercises

    1. Mine and audit hard negatives.
2. Compare contrastive temperatures.
3. Fine-tune a small embedding model and rebuild its index.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
